In [7]:
import pandas as pd
import re

In [8]:
# Load files
pred = pd.read_csv("text_extraction_final.csv")
gt = pd.read_csv("text_extraction_manual_labels_final.csv")

# Merge on ID
df = pred.merge(gt, on="ID", suffixes=("_pred", "_gt"))

# Columns to evaluate
ignore_cols = {"ID", "TargetSurvey"}
base_cols = [c for c in pred.columns if c not in ignore_cols]

In [9]:
def normalize_parish(x):
    if pd.isna(x):
        return x
    return re.sub(r"[ .]", "", str(x).lower())

In [10]:
results = {}

total_correct = 0
total_count = 0

In [11]:
def normalize_numeric(x):
    if pd.isna(x):
        return x
    try:
        return float(x)
    except ValueError:
        return x

In [ ]:
for col in base_cols:
    pred_col = f"{col}_pred"
    gt_col = f"{col}_gt"

    if pred_col not in df or gt_col not in df:
        continue

    gt_vals = df[gt_col]
    pred_vals = df[pred_col]

    if col.lower() == "parish":
        gt_vals = gt_vals.apply(normalize_parish)
        pred_vals = pred_vals.apply(normalize_parish)

    if col.lower() == "total area":
        gt_vals = gt_vals.apply(normalize_numeric)
        pred_vals = pred_vals.apply(normalize_numeric)

    mask = gt_vals.notna()

    correct = (pred_vals[mask] == gt_vals[mask]).sum()
    count = mask.sum()

    accuracy = correct / count if count > 0 else None

    results[col] = {
        "accuracy": accuracy,
        "correct": int(correct),
        "total": int(count)
    }

    total_correct += correct
    total_count += count

In [ ]:
# Overall accuracy
overall_accuracy = total_correct / total_count if total_count > 0 else None

print("Per-column accuracy:")
for col, stats in results.items():
    print(f"{col:25s}: {stats['accuracy']:.4f} ({stats['correct']}/{stats['total']})")

print("\nOverall accuracy:")
print(f"{overall_accuracy:.4f} ({total_correct}/{total_count})")

Per-column accuracy:
Certified date           : 0.9500 (57/60)
Total Area               : 0.9667 (58/60)
Unit of Measurement      : 1.0000 (60/60)
Parish                   : 0.9000 (54/60)
LT Num                   : 0.9167 (55/60)

Overall accuracy:
0.9467 (284/300)
